# Stage 4c — One-Class SVM (OC-SVM)
### Alertreck · Classical Anomaly Detection Paradigm

---

OC-SVM learns a tight decision boundary around **background audio** in feature space. At inference,
clips that fall outside the boundary produce a negative decision score and are flagged as anomalous
(potential threats).

This model provides a **classical ML baseline** for the anomaly detection paradigm alongside the
Conv-AE (Stage 4b), enabling a direct deep-vs-classical comparison within the same evaluation framework.

---

## Feature representation

Raw MFCC shards contain `(N, 120, T)` arrays — 40 MFCCs + 40 Δ + 40 ΔΔ across T time frames.  
Each sample is summarised into a **240-dimensional feature vector**: `[mean_per_coeff ‖ std_per_coeff]`.  
Features are standardised with `StandardScaler` fit on background training data only.

---

## Training strategy

| Setting | Value |
|---|---|
| Training data | Background classes only (`background_animals`, `background_wind_rain`) |
| Feature extraction | Mean + std of MFCC+Δ+ΔΔ coefficients over time → 240-dim |
| Kernel | RBF |
| nu | 0.05 (≈5% of training points may be support vectors / outliers) |
| gamma | scale (1 / (n\_features × X.var())) |
| Threshold | 5th percentile of val background decision scores |

---

## Anomaly detection

1. Extract mean+std features from all background training shards.
2. Standardise features (`StandardScaler` fit on BG train only).
3. Fit `OneClassSVM` on background training features.
4. Score background **validation** set → threshold = 5th-percentile of decision scores.
5. At test time: `decision_function(x) < threshold` → anomalous (potential threat).

---

## Outputs

| File | Description |
|---|---|
| `oc_svm.joblib` | Fitted OneClassSVM |
| `scaler.joblib` | StandardScaler for feature normalisation |
| `score_distributions.png` | Decision score violin plot + histogram per class |
| `roc_pr_curves.png` | Per-threat-class ROC and PR curves |
| `results.json` | Full metrics |
| `model_config_oc_svm.json` | Config + threshold for deployment |

---

**Kaggle dataset:** `orpheusmanga/alertreck-mfcc` · **Output dir:** `/kaggle/working/oc_svm`

> **Note:** If you uploaded MFCC shards inside the same dataset as mel shards, update `data_root` in
> Cell 2 to point to the correct mount path (e.g. `.../alertreck-mel2/mfcc`).

In [4]:
import json
import time
from collections import Counter
from pathlib import Path

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from tqdm import tqdm

matplotlib.use('Agg')
print('Imports OK')

Imports OK


In [5]:
CFG = {
    # ── paths ─────────────────────────────────────────────────────────────────
    # Update mountSlug / sub-path to match your Kaggle dataset attachment.
    'data_root':          Path('/kaggle/input/datasets/orpheusmanga/mfcc-files/mfcc'),
    'output_dir':         Path('/kaggle/working/oc_svm'),

    # ── feature shape ─────────────────────────────────────────────────────────
    'n_mfcc':             40,    # coefficients per frame
    # Shards contain MFCC + delta + delta2  → 120 rows; mean+std → 240-dim vector
    'n_features':         240,

    # ── labels ────────────────────────────────────────────────────────────────
    'label_names': [
        'background_animals',
        'background_wind_rain',
        'threat_chainsaw',
        'threat_dog',
        'threat_gunshot',
        'threat_human',
        'threat_vehicle',
    ],
    'bg_classes':         [0, 1],   # indices that are background

    # ── OC-SVM hyper-parameters ───────────────────────────────────────────────
    'svm_kernel':         'rbf',
    'svm_nu':             0.05,    # upper bound on fraction of outliers in training
    'svm_gamma':          'scale', # 1 / (n_features * X.var())

    # ── anomaly threshold ─────────────────────────────────────────────────────
    # Set at the (100 - anomaly_percentile)th percentile of val BG decision scores.
    # Scores below threshold are flagged anomalous.
    # anomaly_percentile=95 → 95% of BG val passes (FPR ≈ 5% on val).
    'anomaly_percentile': 95,

    'seed': 42,
}

np.random.seed(CFG['seed'])
output_dir = CFG['output_dir']
output_dir.mkdir(parents=True, exist_ok=True)
data_root = CFG['data_root']

print(f'data_root : {data_root}')
print(f'output_dir: {output_dir}')
print(f'Kernel    : {CFG["svm_kernel"]}  nu={CFG["svm_nu"]}  gamma={CFG["svm_gamma"]}')
print(f'Features  : mean+std per MFCC+Δ+ΔΔ coefficient → {CFG["n_features"]}-dim')

data_root : /kaggle/input/datasets/orpheusmanga/mfcc-files/mfcc
output_dir: /kaggle/working/oc_svm
Kernel    : rbf  nu=0.05  gamma=scale
Features  : mean+std per MFCC+Δ+ΔΔ coefficient → 240-dim


In [6]:
# ── Verify all split directories exist ────────────────────────────────────────
for split in ('train', 'val', 'test'):
    d = data_root / split
    n = len(list(d.glob('*.npz'))) if d.exists() else 0
    status = 'OK' if n > 0 else 'MISSING'
    print(f'  {split:<12} {status}  ({n} shards)')


def load_shards(shard_dir: Path) -> tuple:
    """Load all .npz shards; return X (N, 120, T) float32 and y (N,) int64."""
    shards = sorted(shard_dir.glob('*.npz'))
    assert shards, f'No shards in {shard_dir}'
    xs, ys = [], []
    for s in tqdm(shards, desc=f'  {shard_dir.name}'):
        d = np.load(s)
        xs.append(d['X'].astype(np.float32))
        ys.append(d['y'].astype(np.int64))
    X = np.concatenate(xs, axis=0)
    y = np.concatenate(ys, axis=0)
    print(f'  {shard_dir.name}: {X.shape[0]:,} samples  X={X.shape}')
    return X, y


def extract_features(X: np.ndarray) -> np.ndarray:
    """Summarise (N, 120, T) into (N, 240) by concatenating per-coefficient mean and std over time."""
    return np.concatenate([X.mean(axis=2), X.std(axis=2)], axis=1).astype(np.float32)


print('\nLoading shards ...')
X_tr_raw, y_tr = load_shards(data_root / 'train')
X_va_raw, y_va = load_shards(data_root / 'val')
X_te_raw, y_te = load_shards(data_root / 'test')

print('\nExtracting features ...')
F_tr = extract_features(X_tr_raw); del X_tr_raw
F_va = extract_features(X_va_raw); del X_va_raw
F_te = extract_features(X_te_raw); del X_te_raw

bg           = set(CFG['bg_classes'])
tr_bg_mask   = np.isin(y_tr, list(bg))
va_bg_mask   = np.isin(y_va, list(bg))
F_tr_bg      = F_tr[tr_bg_mask]
F_va_bg      = F_va[va_bg_mask]

print(f'\nFeature dim   : {F_tr.shape[1]}  (mean+std of MFCC+Δ+ΔΔ → 2 × 120)')
print(f'BG train size : {F_tr_bg.shape[0]:,}')
print(f'BG val size   : {F_va_bg.shape[0]:,}')
print(f'Test size     : {F_te.shape[0]:,} (all classes)')

print('\nTest class distribution:')
for c, n in sorted(Counter(y_te.tolist()).items()):
    print(f'  {CFG["label_names"][c]:<30} {n:>5}')

  train        OK  (11 shards)
  val          OK  (4 shards)
  test         OK  (4 shards)

Loading shards ...


  train: 100%|██████████| 11/11 [00:17<00:00,  1.60s/it]


  train: 10,223 samples  X=(10223, 120, 301)


  val: 100%|██████████| 4/4 [00:05<00:00,  1.39s/it]


  val: 3,392 samples  X=(3392, 120, 301)


  test: 100%|██████████| 4/4 [00:04<00:00,  1.22s/it]


  test: 3,439 samples  X=(3439, 120, 301)

Extracting features ...

Feature dim   : 240  (mean+std of MFCC+Δ+ΔΔ → 2 × 120)
BG train size : 4,254
BG val size   : 1,414
Test size     : 3,439 (all classes)

Test class distribution:
  background_animals              1190
  background_wind_rain             272
  threat_chainsaw                  513
  threat_dog                       214
  threat_gunshot                   480
  threat_human                     551
  threat_vehicle                   219


In [7]:
print('Standardising features (StandardScaler fit on BG train) ...')
scaler    = StandardScaler()
F_tr_bg_s = scaler.fit_transform(F_tr_bg)
F_va_bg_s = scaler.transform(F_va_bg)
F_te_s    = scaler.transform(F_te)

print(f'\nFitting OneClassSVM ...')
print(f'  kernel={CFG["svm_kernel"]}  nu={CFG["svm_nu"]}  gamma={CFG["svm_gamma"]}')
t0  = time.time()
svm = OneClassSVM(
    kernel=CFG['svm_kernel'],
    nu=CFG['svm_nu'],
    gamma=CFG['svm_gamma'],
    verbose=False,
)
svm.fit(F_tr_bg_s)
elapsed = time.time() - t0

n_sv = len(svm.support_vectors_)
print(f'  Fit complete in {elapsed:.1f}s')
print(f'  Support vectors: {n_sv:,} / {F_tr_bg_s.shape[0]:,} '
      f'({100 * n_sv / F_tr_bg_s.shape[0]:.1f}%)')

Standardising features (StandardScaler fit on BG train) ...

Fitting OneClassSVM ...
  kernel=rbf  nu=0.05  gamma=scale
  Fit complete in 0.2s
  Support vectors: 285 / 4,254 (6.7%)


In [8]:
# decision_function: high score -> inlier (background), low/negative -> outlier (threat)
va_bg_df  = svm.decision_function(F_va_bg_s)

# threshold = (100 - anomaly_percentile)th percentile of BG val scores
# Samples with score < threshold are flagged anomalous.
pct_lower = 100 - CFG['anomaly_percentile']
threshold = float(np.percentile(va_bg_df, pct_lower))

print('Val BG decision scores:')
print(f'  mean  = {va_bg_df.mean():.4f}')
print(f'  std   = {va_bg_df.std():.4f}')
print(f'  min   = {va_bg_df.min():.4f}')
print(f'  max   = {va_bg_df.max():.4f}')
print(f'\nAnomaly threshold (p{pct_lower}): {threshold:.4f}')
print(f'  -> {CFG["anomaly_percentile"]}% of val BG samples score above threshold')

Val BG decision scores:
  mean  = 2.8288
  std   = 1.6582
  min   = -4.2986
  max   = 6.7341

Anomaly threshold (p5): -0.4576
  -> 95% of val BG samples score above threshold


In [9]:
te_df          = svm.decision_function(F_te_s)
anomaly_scores = -te_df                       # higher -> more anomalous (for AUC / ROC)
binary_labels  = (y_te >= 2).astype(int)      # 0 = background, 1 = threat
bg_te_mask     = y_te < 2

# ── Binary metrics ────────────────────────────────────────────────────────────
binary_auc = roc_auc_score(binary_labels, anomaly_scores)
binary_ap  = average_precision_score(binary_labels, anomaly_scores)

preds      = (te_df < threshold).astype(int)  # 1 = flagged anomalous
binary_acc = float((preds == binary_labels).mean())

tp  = int(((preds == 1) & (binary_labels == 1)).sum())
fp  = int(((preds == 1) & (binary_labels == 0)).sum())
fn  = int(((preds == 0) & (binary_labels == 1)).sum())
tn  = int(((preds == 0) & (binary_labels == 0)).sum())
tpr = tp / (tp + fn + 1e-8)
fpr = fp / (fp + tn + 1e-8)

print('Binary anomaly detection (threats vs backgrounds):')
print(f'  AUC-ROC  : {binary_auc:.4f}')
print(f'  Avg Prec : {binary_ap:.4f}')
print(f'  Accuracy : {binary_acc:.4f}')
print(f'  TPR      : {tpr:.4f}')
print(f'  FPR      : {fpr:.4f}')
print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')

# ── Per-class AUC (each threat class vs all background test samples) ──────────
label_names    = CFG['label_names']
per_class_auc  = {}
per_class_ap   = {}
for c in range(2, 7):
    mask  = bg_te_mask | (y_te == c)
    y_bin = (y_te[mask] == c).astype(int)
    s     = anomaly_scores[mask]
    try:
        per_class_auc[label_names[c]] = float(roc_auc_score(y_bin, s))
        per_class_ap[label_names[c]]  = float(average_precision_score(y_bin, s))
    except Exception:
        per_class_auc[label_names[c]] = float('nan')
        per_class_ap[label_names[c]]  = float('nan')

print('\nPer-class AUC-ROC (threat vs background):')
for name in per_class_auc:
    print(f'  {name:<30} AUC={per_class_auc[name]:.4f}  AP={per_class_ap[name]:.4f}')

Binary anomaly detection (threats vs backgrounds):
  AUC-ROC  : 0.7790
  Avg Prec : 0.7822
  Accuracy : 0.5100
  TPR      : 0.1958
  FPR      : 0.0650
  TP=387  FP=95  FN=1590  TN=1367

Per-class AUC-ROC (threat vs background):
  threat_chainsaw                AUC=0.8126  AP=0.5185
  threat_dog                     AUC=0.5907  AP=0.1701
  threat_gunshot                 AUC=0.8050  AP=0.4381
  threat_human                   AUC=0.8321  AP=0.6460
  threat_vehicle                 AUC=0.6935  AP=0.1876


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: violin per class
ax = axes[0]
data_by_class = [te_df[y_te == c] for c in range(7)]
colors = ['#4CAF50' if c < 2 else '#F44336' for c in range(7)]
parts = ax.violinplot(data_by_class, positions=range(7), showmedians=True, showextrema=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(colors[i])
    pc.set_alpha(0.55)
ax.axhline(threshold, color='black', linestyle='--', linewidth=1.5,
           label=f'Threshold ({threshold:.4f})')
ax.set_xticks(range(7))
ax.set_xticklabels([n.replace('_', '\n') for n in label_names], fontsize=8)
ax.set_ylabel('OC-SVM Decision Score')
ax.set_title('Decision Score Distribution by Class\n(green = background, red = threat)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Right: histogram BG vs threat
ax = axes[1]
bg_scores  = te_df[y_te < 2]
thr_scores = te_df[y_te >= 2]
bins = np.linspace(te_df.min(), te_df.max(), 50)
ax.hist(bg_scores,  bins=bins, alpha=0.6, color='#4CAF50',
        label=f'Background (n={len(bg_scores):,})', density=True)
ax.hist(thr_scores, bins=bins, alpha=0.6, color='#F44336',
        label=f'Threat (n={len(thr_scores):,})',     density=True)
ax.axvline(threshold, color='black', linestyle='--', linewidth=1.5, label='Threshold')
ax.set_xlabel('Decision Score')
ax.set_ylabel('Density')
ax.set_title('Background vs Threat Score Distributions')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved score_distributions.png')

Saved score_distributions.png


In [11]:
threat_names = [n for n in label_names if n.startswith('threat')]
colors_t     = plt.cm.Set2(np.linspace(0, 1, len(threat_names)))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
for name, col in zip(threat_names, colors_t):
    c    = label_names.index(name)
    mask = bg_te_mask | (y_te == c)
    y_b  = (y_te[mask] == c).astype(int)
    fpr_v, tpr_v, _ = roc_curve(y_b, anomaly_scores[mask])
    ax.plot(fpr_v, tpr_v, color=col,
            label=f'{name} ({per_class_auc[name]:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('ROC Curves (per threat class vs background)')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

ax = axes[1]
for name, col in zip(threat_names, colors_t):
    c    = label_names.index(name)
    mask = bg_te_mask | (y_te == c)
    y_b  = (y_te[mask] == c).astype(int)
    prec_v, rec_v, _ = precision_recall_curve(y_b, anomaly_scores[mask])
    ax.plot(rec_v, prec_v, color=col,
            label=f'{name} ({per_class_ap[name]:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('PR Curves (per threat class vs background)')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved roc_pr_curves.png')

Saved roc_pr_curves.png


In [12]:
joblib.dump(svm,    output_dir / 'oc_svm.joblib')
joblib.dump(scaler, output_dir / 'scaler.joblib')

results = {
    'model':              'OC-SVM',
    'n_features':         int(F_tr.shape[1]),
    'n_train_bg':         int(F_tr_bg.shape[0]),
    'n_support_vectors':  int(len(svm.support_vectors_)),
    'svm_kernel':         CFG['svm_kernel'],
    'svm_nu':             CFG['svm_nu'],
    'svm_gamma':          CFG['svm_gamma'],
    'anomaly_threshold':  threshold,
    'anomaly_percentile': CFG['anomaly_percentile'],
    'binary_auc':         float(binary_auc),
    'binary_ap':          float(binary_ap),
    'binary_acc':         binary_acc,
    'tpr_at_threshold':   float(tpr),
    'fpr_at_threshold':   float(fpr),
    'confusion':          {'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn},
    'per_class_auc':      per_class_auc,
    'per_class_ap':       per_class_ap,
}

with open(output_dir / 'results.json', 'w') as f:
    json.dump(results, f, indent=2)

model_cfg = {
    'model':             'OC-SVM',
    'n_mfcc':            CFG['n_mfcc'],
    'n_features':        int(F_tr.shape[1]),
    'feature_type':      'mean+std of MFCC+delta+delta2 over time',
    'label_names':       CFG['label_names'],
    'bg_classes':        CFG['bg_classes'],
    'svm_kernel':        CFG['svm_kernel'],
    'svm_nu':            CFG['svm_nu'],
    'svm_gamma':         CFG['svm_gamma'],
    'anomaly_threshold': threshold,
    'anomaly_percentile': CFG['anomaly_percentile'],
    'binary_auc':        float(binary_auc),
}

with open(output_dir / 'model_config_oc_svm.json', 'w') as f:
    json.dump(model_cfg, f, indent=2)

print('Saved:')
print('  oc_svm.joblib')
print('  scaler.joblib')
print('  results.json')
print('  model_config_oc_svm.json')
sep = '─' * 45
print(f'\n{sep}')
print(f'  Binary AUC-ROC   : {binary_auc:.4f}')
print(f'  Binary Avg Prec  : {binary_ap:.4f}')
print(f'  Accuracy         : {binary_acc:.4f}')
print(f'  TPR              : {tpr:.4f}')
print(f'  FPR              : {fpr:.4f}')
print(f'{sep}')

Saved:
  oc_svm.joblib
  scaler.joblib
  results.json
  model_config_oc_svm.json

─────────────────────────────────────────────
  Binary AUC-ROC   : 0.7790
  Binary Avg Prec  : 0.7822
  Accuracy         : 0.5100
  TPR              : 0.1958
  FPR              : 0.0650
─────────────────────────────────────────────
